# 🏭 Notebook 02 — PatchCore Pipeline
## Unsupervised Industrial Defect Detection | MVTec AD Benchmark

**Method:** PatchCore — Memory-Bank Anomaly Detection (No Training Required)  
**Backbone:** WideResNet-50 (ImageNet pretrained, frozen)  
**Categories:** leather · tile · metal_nut  
**Prerequisite:** Notebook 00 + 01 completed. `config.json` and `val_indices.json` must exist on Drive.

---

### STEP 0 — Imports & Bootstrap

In [1]:
# ─────────────────────────────────────────────────────────────────
# PRE-STEP — Install missing packages (run once per session)
# faiss-cpu dan pytorch-msssim tidak ada di Colab default environment
# ─────────────────────────────────────────────────────────────────

import importlib, subprocess, sys

def install_if_missing(module_name: str, pip_name: str):
    if importlib.util.find_spec(module_name) is None:
        print(f"📦 Installing {pip_name}...")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", pip_name],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            raise RuntimeError(f"❌ Failed to install {pip_name}:\n{result.stderr}")
        print(f"  ✅ {pip_name} installed.")
    else:
        print(f"  ⏭️  {pip_name} already available.")

install_if_missing("faiss",          "faiss-cpu")
install_if_missing("pytorch_msssim", "pytorch-msssim")
install_if_missing("psutil",         "psutil")

print("\n✅ Dependencies ready.")

📦 Installing faiss-cpu...
  ✅ faiss-cpu installed.
📦 Installing pytorch-msssim...
  ✅ pytorch-msssim installed.
  ⏭️  psutil already available.

✅ Dependencies ready.


In [2]:
# ─────────────────────────────────────────────────────────────────
# STEP 0 — Imports
# ─────────────────────────────────────────────────────────────────

import os, json, gc, random, time, warnings
import numpy as np
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import cv2
import faiss
import psutil

warnings.filterwarnings('ignore')
print('✅ Imports complete.')

✅ Imports complete.


---
### STEP 1 — Load Config + Set Device

In [3]:
# ─────────────────────────────────────────────────────────────────
# STEP 1 — Load config + device setup
# ─────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_NAME = 'industrial_anomaly_project'
DRIVE_ROOT   = f'/content/drive/MyDrive/{PROJECT_NAME}'
CONFIG_PATH  = f'{DRIVE_ROOT}/configs/config.json'

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(
        f'❌ config.json not found. Run Notebook 00 first.\n{CONFIG_PATH}'
    )

with open(CONFIG_PATH) as f:
    cfg = json.load(f)

# ── Unpack paths ──────────────────────────────────────────────────
DATASET_DRIVE  = cfg['DATASET_DRIVE']
MODEL_DRIVE    = cfg['MODEL_DRIVE']
RESULTS_DRIVE  = cfg['RESULTS_DRIVE']
LOGS_DRIVE     = cfg['LOGS_DRIVE']
CONFIG_DRIVE   = cfg['CONFIG_DRIVE']
LOCAL_ROOT     = cfg['LOCAL_ROOT']
LOCAL_DATA     = cfg['LOCAL_DATA']
LOCAL_TEMP     = cfg['LOCAL_TEMP']
LOCAL_MODELS   = cfg['LOCAL_MODELS']
CATEGORIES     = cfg['CATEGORIES']
SEED           = cfg['SEED']
IMAGE_SIZE     = cfg['IMAGE_SIZE']
IMAGENET_MEAN  = cfg['IMAGENET_MEAN']
IMAGENET_STD   = cfg['IMAGENET_STD']
VAL_SPLIT      = cfg['VAL_SPLIT']

# PatchCore hyperparams
CORESET_RATIO  = cfg['PATCHCORE_CORESET_RATIO']
KNN_K          = cfg['PATCHCORE_KNN_K']

# ── Recreate local dirs ───────────────────────────────────────────
for d in [LOCAL_ROOT, LOCAL_DATA, LOCAL_TEMP, LOCAL_MODELS]:
    os.makedirs(d, exist_ok=True)
os.makedirs(MODEL_DRIVE, exist_ok=True)
os.makedirs(RESULTS_DRIVE, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── Device ────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    gpu_name  = torch.cuda.get_device_name(0)
    gpu_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'✅ GPU: {gpu_name} | VRAM: {gpu_total:.2f} GB')
else:
    DEVICE = torch.device('cpu')
    print('⚠️  No GPU detected. PatchCore will be very slow on CPU.')

print(f'   Device : {DEVICE}')
print(f'   Seed   : {SEED}')
print(f'   Categories: {CATEGORIES}')

# ── Memory utils ─────────────────────────────────────────────────
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def print_mem():
    vm = psutil.virtual_memory()
    ram = f'{vm.used/1e9:.1f}/{vm.total/1e9:.1f} GB ({vm.percent:.0f}%)'
    if torch.cuda.is_available():
        vram_used  = torch.cuda.memory_allocated(0)/1e9
        vram_total = torch.cuda.get_device_properties(0).total_memory/1e9
        print(f'   RAM: {ram} | VRAM: {vram_used:.2f}/{vram_total:.2f} GB')
    else:
        print(f'   RAM: {ram}')

Mounted at /content/drive
✅ GPU: Tesla T4 | VRAM: 14.56 GB
   Device : cuda
   Seed   : 42
   Categories: ['leather', 'tile', 'metal_nut']


In [4]:
# ─────────────────────────────────────────────────────────────────
# STEP 1b — Restore local dataset from Drive (run after every session reset)
#
# /content/ adalah ephemeral storage — hilang saat session Colab reset.
# Cell ini mengecek apakah data sudah ada di LOCAL_DATA.
# Jika belum, copy ulang dari Drive (source of truth).
# ─────────────────────────────────────────────────────────────────

import shutil, time

print("🔍 Checking local dataset availability...\n")

for cat in CATEGORIES:
    src  = os.path.join(DATASET_DRIVE, cat)
    dest = os.path.join(LOCAL_DATA, cat)

    # Cek apakah folder exist DAN tidak kosong
    dest_ok = (
        os.path.exists(dest) and
        os.path.exists(os.path.join(dest, "train", "good")) and
        len(os.listdir(os.path.join(dest, "train", "good"))) > 0
    )

    if dest_ok:
        n_train = len(os.listdir(os.path.join(dest, "train", "good")))
        print(f"  ⏭️  {cat:<12}: already at LOCAL_DATA ({n_train} train images) — skipped.")
        continue

    # Source harus ada di Drive
    if not os.path.exists(src):
        raise FileNotFoundError(
            f"❌ Dataset not found on Drive: {src}\n"
            f"Re-run Notebook 01 to download the dataset."
        )

    # Copy Drive → /content/
    t0 = time.time()
    print(f"  📋 {cat:<12}: copying from Drive...", end="", flush=True)
    if os.path.exists(dest):
        shutil.rmtree(dest)  # remove partial/corrupt local copy
    shutil.copytree(src, dest)
    elapsed = time.time() - t0
    n_train = len(os.listdir(os.path.join(dest, "train", "good")))
    print(f" done in {elapsed:.1f}s ({n_train} train images)")

print("\n✅ All categories available at LOCAL_DATA.")

🔍 Checking local dataset availability...

  📋 leather     : copying from Drive... done in 92.5s (245 train images)
  📋 tile        : copying from Drive... done in 73.5s (230 train images)
  📋 metal_nut   : copying from Drive... done in 73.6s (220 train images)

✅ All categories available at LOCAL_DATA.


---
### STEP 2 — Dataset Class + Transform

In [5]:
# ─────────────────────────────────────────────────────────────────
# STEP 2 — MVTecDataset + transforms
# ─────────────────────────────────────────────────────────────────

IMG_EXTENSIONS = {'.png', '.jpg', '.jpeg'}

# ── Transforms ───────────────────────────────────────────────────
TRANSFORM_IMG = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

TRANSFORM_MASK = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=T.InterpolationMode.NEAREST),
    T.ToTensor(),
])


class MVTecDataset(Dataset):
    """
    MVTec AD Dataset loader.

    train mode : returns (image, label=0, mask=zeros, defect_type='good')
    test  mode : returns (image, label, mask, defect_type)
                 label  = 0 (normal) | 1 (anomaly)
                 mask   = binary pixel mask (zeros for normal)
    """

    def __init__(self, root: str, category: str, split: str,
                 transform_img=None, transform_mask=None,
                 exclude_paths: set = None):
        self.root           = Path(root) / category
        self.split          = split
        self.transform_img  = transform_img  or TRANSFORM_IMG
        self.transform_mask = transform_mask or TRANSFORM_MASK
        self.exclude_paths  = exclude_paths or set()
        self.samples        = []  # list of (img_path, label, defect_type, mask_path|None)
        self._build_samples()

    def _build_samples(self):
        split_dir = self.root / self.split
        gt_dir    = self.root / 'ground_truth'

        if self.split == 'train':
            # Train: only normal images
            good_dir = split_dir / 'good'
            for p in sorted(good_dir.iterdir()):
                if p.suffix.lower() in IMG_EXTENSIONS:
                    if str(p) not in self.exclude_paths:
                        self.samples.append((p, 0, 'good', None))
        else:
            # Test: all subfolders
            for defect_dir in sorted(split_dir.iterdir()):
                if not defect_dir.is_dir():
                    continue
                is_normal    = defect_dir.name == 'good'
                label        = 0 if is_normal else 1
                defect_type  = defect_dir.name

                for p in sorted(defect_dir.iterdir()):
                    if p.suffix.lower() not in IMG_EXTENSIONS:
                        continue
                    if str(p) in self.exclude_paths:
                        continue
                    if is_normal:
                        mask_path = None
                    else:
                        mask_path = gt_dir / defect_type / f'{p.stem}_mask{p.suffix}'
                        if not mask_path.exists():
                            mask_path = None
                    self.samples.append((p, label, defect_type, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label, defect_type, mask_path = self.samples[idx]

        # Image
        img = Image.open(img_path).convert('RGB')
        img = self.transform_img(img)  # (3, H, W)

        # Mask
        if mask_path is not None and mask_path.exists():
            mask = Image.open(mask_path).convert('L')
            mask = self.transform_mask(mask)  # (1, H, W)
            mask = (mask > 0.5).float()
        else:
            mask = torch.zeros(1, IMAGE_SIZE, IMAGE_SIZE)

        return img, torch.tensor(label, dtype=torch.long), mask, defect_type, str(img_path)


print('✅ MVTecDataset defined.')

✅ MVTecDataset defined.


In [6]:
# ─────────────────────────────────────────────────────────────────
# STEP 2b — Sanity check: load 1 batch from leather train
# ─────────────────────────────────────────────────────────────────

_ds = MVTecDataset(LOCAL_DATA, 'leather', 'train')
_dl = DataLoader(_ds, batch_size=4, shuffle=False, num_workers=2)
_batch = next(iter(_dl))
_img, _lbl, _mask, _dtype, _paths = _batch

print('Sanity check — leather train batch:')
print(f'  image shape  : {_img.shape}   dtype: {_img.dtype}')
print(f'  label shape  : {_lbl.shape}   values: {_lbl.tolist()}')
print(f'  mask shape   : {_mask.shape}  max: {_mask.max().item()}')
print(f'  defect types : {list(_dtype)}')
print(f'  pixel range  : [{_img.min():.3f}, {_img.max():.3f}]')

assert _img.shape == (4, 3, IMAGE_SIZE, IMAGE_SIZE), 'Image shape mismatch'
assert _lbl.sum().item() == 0, 'Train labels should all be 0 (normal)'
print('\n✅ Sanity check passed.')

del _ds, _dl, _batch, _img, _lbl, _mask
clear_memory()

Sanity check — leather train batch:
  image shape  : torch.Size([4, 3, 256, 256])   dtype: torch.float32
  label shape  : torch.Size([4])   values: [0, 0, 0, 0]
  mask shape   : torch.Size([4, 1, 256, 256])  max: 0.0
  defect types : ['good', 'good', 'good', 'good']
  pixel range  : [-1.335, 0.605]

✅ Sanity check passed.


---
### STEP 3 — Load Validation Split (LOCKED)

In [7]:
# ─────────────────────────────────────────────────────────────────
# STEP 3 — Load val_indices.json (generated & locked in Notebook 01)
#
# val_indices is used ONLY for threshold selection in Notebook 04.
# Here we load it to build exclude_paths for test DataLoader,
# ensuring val images are excluded from scoring evaluation.
# ─────────────────────────────────────────────────────────────────

VAL_INDICES_PATH = f'{CONFIG_DRIVE}/val_indices.json'

if not os.path.exists(VAL_INDICES_PATH):
    raise FileNotFoundError(
        f'❌ val_indices.json not found at: {VAL_INDICES_PATH}\n'
        f'Run the pre-notebook fixes in Notebook 01 first.'
    )

with open(VAL_INDICES_PATH) as f:
    VAL_INDICES = json.load(f)

# Build per-category set of validation image paths
# These paths are LOCAL_DATA based — remap if needed
def get_val_exclude_set(category: str) -> set:
    """
    Returns a set of LOCAL_DATA paths for validation images.
    These are excluded from the test holdout DataLoader.
    """
    val_paths = (
        VAL_INDICES[category]['normal'] +
        VAL_INDICES[category]['defective']
    )
    # Normalize to LOCAL_DATA paths (val_indices was created from LOCAL_DATA)
    normalized = set()
    for p in val_paths:
        # Replace any Drive prefix with LOCAL_DATA prefix
        p_path = Path(p)
        # Find the part after category name
        try:
            parts = p_path.parts
            cat_idx = next(i for i, part in enumerate(parts) if part == category)
            relative = Path(*parts[cat_idx:])
            normalized.add(str(Path(LOCAL_DATA) / relative))
        except StopIteration:
            normalized.add(str(p))
    return normalized

print('✅ val_indices.json loaded.')
print()
for cat in CATEGORIES:
    excl = get_val_exclude_set(cat)
    n_normal  = VAL_INDICES[cat]['counts']['normal']
    n_defect  = VAL_INDICES[cat]['counts']['defective']
    print(f'  {cat:<12}: {n_normal} normal + {n_defect} defective excluded from test scoring')

✅ val_indices.json loaded.

  leather     : 10 normal + 10 defective excluded from test scoring
  tile        : 10 normal + 10 defective excluded from test scoring
  metal_nut   : 5 normal + 10 defective excluded from test scoring


---
### STEP 4 — Load Feature Extractor (WideResNet-50)

In [8]:
# ─────────────────────────────────────────────────────────────────
# STEP 4 — WideResNet-50 feature extractor with forward hooks
#
# - All parameters frozen (no gradient computation)
# - Hooks registered on layer2 (512ch) and layer3 (1024ch)
# - Model stays in eval mode permanently
# ─────────────────────────────────────────────────────────────────

class PatchCoreExtractor:
    """
    WideResNet-50 feature extractor for PatchCore.
    Extracts intermediate feature maps via forward hooks.
    """

    def __init__(self, device):
        self.device   = device
        self.features = {}  # filled by hooks
        self._build_model()

    def _build_model(self):
        # Load pretrained WideResNet-50
        backbone = models.wide_resnet50_2(
            weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V1
        )

        # Freeze ALL parameters — no gradient ever computed
        for param in backbone.parameters():
            param.requires_grad = False

        backbone.eval()
        backbone = backbone.to(self.device)

        # Register hooks on layer2 and layer3
        backbone.layer2.register_forward_hook(self._hook('layer2'))
        backbone.layer3.register_forward_hook(self._hook('layer3'))

        self.model = backbone
        print(f'  Backbone   : wide_resnet50_2 (ImageNet pretrained)')
        print(f'  Parameters : {sum(p.numel() for p in backbone.parameters())/1e6:.1f}M (all frozen)')
        print(f'  Hooks      : layer2, layer3')

    def _hook(self, name: str):
        def fn(module, input, output):
            self.features[name] = output
        return fn

    @torch.no_grad()
    def extract(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass → extract + fuse layer2 + layer3 features.

        Args:
            x: (B, 3, H, W) normalized image tensor

        Returns:
            patch_features: (B, N_patches, 1536)
                where N_patches = (H/8) * (W/8)
        """
        self.features.clear()
        _ = self.model(x.to(self.device))

        f2 = self.features['layer2']  # (B, 512,  H/8,  W/8)
        f3 = self.features['layer3']  # (B, 1024, H/16, W/16)

        # Upsample layer3 → match layer2 spatial size
        f3_up = F.interpolate(f3, size=f2.shape[-2:], mode='bilinear', align_corners=False)

        # Concatenate channel-wise → (B, 1536, H/8, W/8)
        fused = torch.cat([f2, f3_up], dim=1)

        # Reshape to (B, N_patches, 1536)
        B, C, H, W = fused.shape
        patch_features = fused.permute(0, 2, 3, 1).reshape(B, H * W, C)

        return patch_features.cpu()  # move to CPU immediately to free VRAM


print('🔧 Loading WideResNet-50...')
extractor = PatchCoreExtractor(DEVICE)
print('✅ Feature extractor ready.')
print_mem()

🔧 Loading WideResNet-50...
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-95faca4d.pth


100%|██████████| 132M/132M [00:00<00:00, 174MB/s]


  Backbone   : wide_resnet50_2 (ImageNet pretrained)
  Parameters : 68.9M (all frozen)
  Hooks      : layer2, layer3
✅ Feature extractor ready.
   RAM: 2.4/13.6 GB (20%) | VRAM: 0.28/15.64 GB


In [9]:
# ── Verify extractor output shape ────────────────────────────────
_dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)
_out   = extractor.extract(_dummy)
print(f'Extractor output shape: {_out.shape}')
# Expected: (2, 32*32, 1536) = (2, 1024, 1536) for 256x256 input
assert _out.shape[0] == 2,    'Batch dim mismatch'
assert _out.shape[2] == 1536, f'Feature dim should be 1536, got {_out.shape[2]}'
PATCH_DIM    = _out.shape[2]   # 1536
N_PATCHES_PER_IMG = _out.shape[1]  # 1024 for 256x256
print(f'  PATCH_DIM         : {PATCH_DIM}')
print(f'  N_PATCHES_PER_IMG : {N_PATCHES_PER_IMG}')
H_PATCH = W_PATCH = int(N_PATCHES_PER_IMG ** 0.5)
print(f'  Patch grid        : {H_PATCH} x {W_PATCH}')
print('✅ Extractor shape verified.')
del _dummy, _out
clear_memory()

Extractor output shape: torch.Size([2, 1024, 1536])
  PATCH_DIM         : 1536
  N_PATCHES_PER_IMG : 1024
  Patch grid        : 32 x 32
✅ Extractor shape verified.


---
### STEP 5 — Feature Extraction (Train Images → memmap)

In [10]:
# ─────────────────────────────────────────────────────────────────
# STEP 5 — extract_features()
#
# Streams patch features from all train/good images to a memmap
# file on local SSD. Never accumulates full matrix in RAM.
#
# Output: np.memmap of shape (N_total_patches, PATCH_DIM)
#         saved at LOCAL_TEMP/patch_features_{category}.dat
# ─────────────────────────────────────────────────────────────────

def extract_features(
    category: str,
    extractor: PatchCoreExtractor,
    batch_size: int = 32,
) -> tuple:
    """
    Extract patch-level features from all train/good images.
    Streams to memmap — safe for large datasets.

    Returns:
        (memmap_path: str, n_total_patches: int)
    """
    dataset = MVTecDataset(LOCAL_DATA, category, 'train')
    loader  = DataLoader(
        dataset, batch_size=batch_size,
        shuffle=False, num_workers=2, pin_memory=True
    )

    n_imgs           = len(dataset)
    n_total_patches  = n_imgs * N_PATCHES_PER_IMG
    memmap_path      = os.path.join(LOCAL_TEMP, f'patch_features_{category}.dat')

    print(f'  [{category}] Train images       : {n_imgs}')
    print(f'  [{category}] Total patches      : {n_total_patches:,}')
    size_mb = n_total_patches * PATCH_DIM * 4 / (1024**2)  # float32
    print(f'  [{category}] memmap size (est.) : {size_mb:.1f} MB')

    # ── Allocate memmap ───────────────────────────────────────────
    mm = np.memmap(
        memmap_path, dtype=np.float32, mode='w+',
        shape=(n_total_patches, PATCH_DIM)
    )

    # ── Stream features batch by batch ───────────────────────────
    write_idx = 0
    t0 = time.time()

    extractor.model.eval()
    for imgs, _, _, _, _ in tqdm(loader, desc=f'  Extracting [{category}]'):
        patches = extractor.extract(imgs)  # (B, N_patches, 1536) on CPU
        B = patches.shape[0]
        flat = patches.numpy().reshape(B * N_PATCHES_PER_IMG, PATCH_DIM)

        mm[write_idx : write_idx + flat.shape[0]] = flat
        write_idx += flat.shape[0]

    mm.flush()  # ensure all writes committed to disk
    elapsed = time.time() - t0

    assert write_idx == n_total_patches, \
        f'Write mismatch: wrote {write_idx}, expected {n_total_patches}'

    print(f'  [{category}] Extraction done in {elapsed:.1f}s')
    del mm  # close memmap handle (file stays on disk)
    clear_memory()

    return memmap_path, n_total_patches


print('✅ extract_features() defined.')

✅ extract_features() defined.


---
### STEP 6 — Coreset Selection (Optimized)

In [11]:
# ─────────────────────────────────────────────────────────────────
# STEP 6 — build_coreset()
#
# Algorithm:
#   1. Random projection 1536 → 128 (Johnson-Lindenstrauss)
#      Purpose: reduce distance computation cost for greedy selection
#   2. Vectorized greedy coreset:
#      - Maintain min_distances[N] array
#      - Each iteration: select argmax, update min_distances
#      - O(N x K) instead of O(N^2)
#   3. Return selected indices (applied to original 1536-dim space)
#
# Memory: operates on projected features (128-dim) to save RAM
# ─────────────────────────────────────────────────────────────────

def build_coreset(
    memmap_path: str,
    n_total: int,
    coreset_ratio: float = 0.10,
    proj_dim: int = 128,
    seed: int = 42,
) -> np.ndarray:
    """
    Greedy coreset subsampling via random projection.

    Args:
        memmap_path   : path to patch features memmap (.dat)
        n_total       : total number of patches
        coreset_ratio : fraction to keep (default 10%)
        proj_dim      : projection dim for distance computation
        seed          : random seed

    Returns:
        selected_indices: np.ndarray of shape (K,) where K = n_total * coreset_ratio
    """
    rng       = np.random.default_rng(seed)
    K         = max(1, int(n_total * coreset_ratio))
    print(f'    Total patches   : {n_total:,}')
    print(f'    Coreset size K  : {K:,} ({coreset_ratio*100:.0f}%)')
    print(f'    Projection dim  : {proj_dim}')

    # ── Load memmap (read-only) ───────────────────────────────────
    mm = np.memmap(memmap_path, dtype=np.float32, mode='r', shape=(n_total, PATCH_DIM))

    # ── Random projection matrix: (PATCH_DIM, proj_dim) ──────────
    # Scale by 1/sqrt(proj_dim) to preserve L2 distances (JL lemma)
    proj_matrix = rng.standard_normal((PATCH_DIM, proj_dim)).astype(np.float32)
    proj_matrix /= np.sqrt(proj_dim)

    # ── Project all features to low-dim space ─────────────────────
    # Process in chunks to avoid RAM spike
    CHUNK = 10000
    projected = np.zeros((n_total, proj_dim), dtype=np.float32)
    for start in range(0, n_total, CHUNK):
        end = min(start + CHUNK, n_total)
        projected[start:end] = mm[start:end] @ proj_matrix

    del mm  # release memmap handle

    # ── Vectorized greedy coreset ─────────────────────────────────
    t0             = time.time()
    selected       = []
    # Seed: pick random first point
    first_idx      = int(rng.integers(0, n_total))
    selected.append(first_idx)

    # min_distances[i] = min distance from point i to any selected point
    min_distances = np.full(n_total, fill_value=np.inf, dtype=np.float32)

    for step in tqdm(range(K - 1), desc='    Greedy coreset'):
        last = projected[selected[-1]]  # (proj_dim,)

        # Vectorized L2 distance from all points to last selected
        diff = projected - last          # (N, proj_dim)
        dists = (diff * diff).sum(axis=1)  # (N,)  — squared L2

        # Update minimum distances
        np.minimum(min_distances, dists, out=min_distances)

        # Select the farthest point from all selected
        next_idx = int(np.argmax(min_distances))
        selected.append(next_idx)
        min_distances[next_idx] = 0.0  # mark as selected

    elapsed = time.time() - t0
    selected_indices = np.array(selected, dtype=np.int64)
    print(f'    Coreset done in {elapsed:.1f}s')

    del projected, proj_matrix
    gc.collect()

    return selected_indices


print('✅ build_coreset() defined.')

✅ build_coreset() defined.


---
### STEP 7 — Build Memory Bank

In [12]:
# ─────────────────────────────────────────────────────────────────
# STEP 7 — build_memory_bank()
#
# Loads coreset indices, subsets the ORIGINAL 1536-dim features,
# saves to Drive as patchcore_{category}.npy
# ─────────────────────────────────────────────────────────────────

def build_memory_bank(
    category: str,
    memmap_path: str,
    n_total: int,
    coreset_indices: np.ndarray,
) -> np.ndarray:
    """
    Subset original 1536-dim features with coreset indices.
    Saves to Drive and returns the memory bank array.
    """
    mm      = np.memmap(memmap_path, dtype=np.float32, mode='r', shape=(n_total, PATCH_DIM))
    bank    = mm[coreset_indices].copy()  # (K, 1536) — load only selected rows
    del mm

    # Save to Drive (persistent)
    drive_path = os.path.join(MODEL_DRIVE, f'patchcore_{category}.npy')
    np.save(drive_path, bank)

    # Save to local (fast access for scoring)
    local_path = os.path.join(LOCAL_MODELS, f'patchcore_{category}.npy')
    np.save(local_path, bank)

    size_mb = bank.nbytes / (1024**2)
    print(f'  [{category}] Memory bank : {bank.shape} | {size_mb:.1f} MB')
    print(f'  [{category}] Saved to Drive : {drive_path}')

    # Delete memmap file — no longer needed
    if os.path.exists(memmap_path):
        os.remove(memmap_path)
        print(f'  [{category}] Temp memmap deleted.')

    return bank


print('✅ build_memory_bank() defined.')

✅ build_memory_bank() defined.


---
### STEP 8 — FAISS Index

In [13]:
# ─────────────────────────────────────────────────────────────────
# STEP 8 — build_faiss_index()
#
# Uses IndexFlatL2 (exact search) for coreset sizes ~10K-25K.
# Falls back to IndexIVFFlat for very large banks (>50K).
# ─────────────────────────────────────────────────────────────────

def build_faiss_index(memory_bank: np.ndarray) -> faiss.Index:
    """
    Build FAISS index from memory bank.

    Args:
        memory_bank: (K, D) float32 array

    Returns:
        faiss.Index ready for search
    """
    bank = memory_bank.astype(np.float32)
    K, D = bank.shape

    if K <= 50000:
        # Exact search — safe and simple for coreset sizes
        index = faiss.IndexFlatL2(D)
        index.add(bank)
        index_type = 'IndexFlatL2 (exact)'
    else:
        # Approximate search for very large banks
        nlist     = min(100, K // 10)
        quantizer = faiss.IndexFlatL2(D)
        index     = faiss.IndexIVFFlat(quantizer, D, nlist)
        index.train(bank)
        index.add(bank)
        index.nprobe = 10
        index_type = f'IndexIVFFlat (nlist={nlist}, approximate)'

    print(f'  FAISS index : {index_type}')
    print(f'  Vectors     : {index.ntotal:,} | dim: {D}')
    return index


print('✅ build_faiss_index() defined.')

✅ build_faiss_index() defined.


---
### STEP 9 — Anomaly Scoring

In [14]:
# ─────────────────────────────────────────────────────────────────
# STEP 9 — compute_scores()
#
# For each test image (excluding validation set):
#   1. Extract patch features
#   2. Query FAISS for k-NN distances
#   3. anomaly_map  = distances reshaped to (H_patch, W_patch)
#   4. anomaly_score = max distance across all patches
#
# Outputs saved to LOCAL_TEMP (moved to RESULTS_DRIVE after loop):
#   scores_{category}.npy   — (N_test,) float32
#   maps_{category}.npy     — (N_test, H_patch, W_patch) float32
#   labels_{category}.npy   — (N_test,) int
#   meta_{category}.json    — paths, defect_types per image
# ─────────────────────────────────────────────────────────────────

def compute_scores(
    category: str,
    extractor: PatchCoreExtractor,
    index: faiss.Index,
    k: int = 9,
    batch_size: int = 16,
) -> dict:
    """
    Score all test holdout images for a category.

    Returns:
        dict with keys: scores, maps, labels, paths, defect_types
    """
    # ── Build test DataLoader (exclude validation images) ─────────
    exclude = get_val_exclude_set(category)
    dataset = MVTecDataset(
        LOCAL_DATA, category, 'test',
        exclude_paths=exclude
    )
    loader = DataLoader(
        dataset, batch_size=batch_size,
        shuffle=False, num_workers=2, pin_memory=True
    )

    print(f'  [{category}] Test holdout images: {len(dataset)}')

    all_scores       = []
    all_maps         = []
    all_labels       = []
    all_paths        = []
    all_defect_types = []

    extractor.model.eval()
    t0 = time.time()

    for imgs, labels, _, defect_types, img_paths in tqdm(
        loader, desc=f'  Scoring [{category}]'
    ):
        # Extract features: (B, N_patches, 1536)
        patches = extractor.extract(imgs).numpy()  # CPU numpy
        B = patches.shape[0]

        for i in range(B):
            p = patches[i]  # (N_patches, 1536)

            # k-NN search
            distances, _ = index.search(p.astype(np.float32), k)
            # distances: (N_patches, k) — squared L2

            # Use nearest neighbor distance (k=1 equivalent for scoring)
            nn_dists = distances[:, 0]  # (N_patches,)

            # Anomaly map: reshape to patch grid
            amap = nn_dists.reshape(H_PATCH, W_PATCH)  # (H_p, W_p)

            # Anomaly score: max distance across all patches
            score = float(nn_dists.max())

            all_scores.append(score)
            all_maps.append(amap)
            all_labels.append(int(labels[i].item()))
            all_paths.append(img_paths[i])
            all_defect_types.append(defect_types[i])

    elapsed = time.time() - t0
    n_test  = len(all_scores)
    print(f'  [{category}] Scoring done in {elapsed:.1f}s ({elapsed/n_test*1000:.1f} ms/img)')

    # ── Convert to arrays ─────────────────────────────────────────
    scores = np.array(all_scores, dtype=np.float32)         # (N,)
    maps   = np.stack(all_maps,   axis=0).astype(np.float32) # (N, H_p, W_p)
    labels = np.array(all_labels, dtype=np.int32)           # (N,)

    return {
        'scores':       scores,
        'maps':         maps,
        'labels':       labels,
        'paths':        all_paths,
        'defect_types': all_defect_types,
    }


print('✅ compute_scores() defined.')

✅ compute_scores() defined.


---
### STEP 10 — Save Results & Cleanup

In [15]:
# ─────────────────────────────────────────────────────────────────
# STEP 10 — save_results() + cleanup helpers
# ─────────────────────────────────────────────────────────────────

def save_results(category: str, results: dict):
    """
    Save scores, maps, labels, and metadata to RESULTS_DRIVE.
    """
    prefix = os.path.join(RESULTS_DRIVE, f'patchcore_{category}')
    os.makedirs(RESULTS_DRIVE, exist_ok=True)

    np.save(f'{prefix}_scores.npy', results['scores'])
    np.save(f'{prefix}_maps.npy',   results['maps'])
    np.save(f'{prefix}_labels.npy', results['labels'])

    meta = {
        'paths':        results['paths'],
        'defect_types': results['defect_types'],
        'n_total':      len(results['scores']),
        'n_normal':     int((results['labels'] == 0).sum()),
        'n_defective':  int((results['labels'] == 1).sum()),
    }
    with open(f'{prefix}_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)

    print(f'  [{category}] Results saved:')
    print(f'     scores  : {results["scores"].shape} | range [{results["scores"].min():.4f}, {results["scores"].max():.4f}]')
    print(f'     maps    : {results["maps"].shape}')
    print(f'     labels  : {results["labels"].shape} | normal={meta["n_normal"]} defective={meta["n_defective"]}')


def cleanup_category(category: str, bank=None, index=None, results=None):
    """Release all large objects after processing one category."""
    del bank, index, results
    # Delete any remaining temp files
    for fname in [f'patch_features_{category}.dat']:
        p = os.path.join(LOCAL_TEMP, fname)
        if os.path.exists(p):
            os.remove(p)
    clear_memory()
    print(f'  [{category}] Memory cleared.')


print('✅ save_results() and cleanup_category() defined.')

✅ save_results() and cleanup_category() defined.


---
### STEP 11 — Main Loop: All Categories

In [16]:
# ─────────────────────────────────────────────────────────────────
# STEP 11 — Main pipeline loop
#
# Processes ONE category at a time:
#   extract → coreset → memory bank → FAISS → score → save → cleanup
#
# Total estimated time on T4:
#   ~20-30 min per category (dominated by feature extraction)
# ─────────────────────────────────────────────────────────────────

pipeline_metadata = {}  # collects per-category stats for STEP 12

for CATEGORY in CATEGORIES:
    print()
    print('═' * 60)
    print(f'  Processing category: {CATEGORY.upper()}')
    print('═' * 60)
    t_cat = time.time()

    # ── 5: Feature Extraction ─────────────────────────────────────
    print('\n[5] Feature extraction...')
    memmap_path, n_total_patches = extract_features(
        CATEGORY, extractor, batch_size=32
    )
    print_mem()

    # ── 6: Coreset Selection ──────────────────────────────────────
    print('\n[6] Coreset selection...')
    coreset_indices = build_coreset(
        memmap_path, n_total_patches,
        coreset_ratio=CORESET_RATIO, seed=SEED
    )
    print_mem()

    # ── 7: Memory Bank ────────────────────────────────────────────
    print('\n[7] Building memory bank...')
    memory_bank = build_memory_bank(
        CATEGORY, memmap_path, n_total_patches, coreset_indices
    )
    print_mem()

    # ── 8: FAISS Index ────────────────────────────────────────────
    print('\n[8] Building FAISS index...')
    faiss_index = build_faiss_index(memory_bank)

    # ── 9: Anomaly Scoring ────────────────────────────────────────
    print('\n[9] Anomaly scoring...')
    results = compute_scores(
        CATEGORY, extractor, faiss_index,
        k=KNN_K, batch_size=16
    )

    # ── 10: Save Results ──────────────────────────────────────────
    print('\n[10] Saving results...')
    save_results(CATEGORY, results)

    # ── Collect metadata ──────────────────────────────────────────
    elapsed_cat = time.time() - t_cat
    pipeline_metadata[CATEGORY] = {
        'n_train_images':      len(MVTecDataset(LOCAL_DATA, CATEGORY, 'train')),
        'n_total_patches':     int(n_total_patches),
        'coreset_size':        int(len(coreset_indices)),
        'coreset_ratio':       CORESET_RATIO,
        'patch_dim':           PATCH_DIM,
        'patch_grid':          f'{H_PATCH}x{W_PATCH}',
        'memory_bank_shape':   list(memory_bank.shape),
        'memory_bank_mb':      round(memory_bank.nbytes / (1024**2), 2),
        'n_test_scored':       int(len(results['scores'])),
        'n_test_normal':       int((results['labels'] == 0).sum()),
        'n_test_defective':    int((results['labels'] == 1).sum()),
        'score_min':           float(results['scores'].min()),
        'score_max':           float(results['scores'].max()),
        'score_mean':          float(results['scores'].mean()),
        'elapsed_seconds':     round(elapsed_cat, 1),
    }

    # ── Cleanup ───────────────────────────────────────────────────
    print(f'\n[Cleanup] {CATEGORY}...')
    cleanup_category(CATEGORY, memory_bank, faiss_index, results)
    del coreset_indices, memmap_path, n_total_patches
    del memory_bank, faiss_index, results
    clear_memory()

    print(f'\n✅ {CATEGORY.upper()} done in {elapsed_cat/60:.1f} min')

print()
print('═' * 60)
print('  All categories processed.')
print('═' * 60)


════════════════════════════════════════════════════════════
  Processing category: LEATHER
════════════════════════════════════════════════════════════

[5] Feature extraction...
  [leather] Train images       : 245
  [leather] Total patches      : 250,880
  [leather] memmap size (est.) : 1470.0 MB


  Extracting [leather]: 100%|██████████| 8/8 [00:17<00:00,  2.23s/it]


  [leather] Extraction done in 18.2s
   RAM: 2.6/13.6 GB (23%) | VRAM: 0.35/15.64 GB

[6] Coreset selection...
    Total patches   : 250,880
    Coreset size K  : 25,088 (10%)
    Projection dim  : 128


    Greedy coreset: 100%|██████████| 25087/25087 [44:46<00:00,  9.34it/s]


    Coreset done in 2686.3s
   RAM: 2.8/13.6 GB (24%) | VRAM: 0.35/15.64 GB

[7] Building memory bank...
  [leather] Memory bank : (25088, 1536) | 147.0 MB
  [leather] Saved to Drive : /content/drive/MyDrive/industrial_anomaly_project/models/patchcore_leather.npy
  [leather] Temp memmap deleted.
   RAM: 2.9/13.6 GB (25%) | VRAM: 0.35/15.64 GB

[8] Building FAISS index...
  FAISS index : IndexFlatL2 (exact)
  Vectors     : 25,088 | dim: 1536

[9] Anomaly scoring...
  [leather] Test holdout images: 104


...

  [metal_nut] Extraction done in 20.9s
   RAM: 2.8/13.6 GB (26%) | VRAM: 0.38/15.64 GB

[6] Coreset selection...
    Total patches   : 225,280
    Coreset size K  : 22,528 (10%)
    Projection dim  : 128


    Greedy coreset: 100%|██████████| 22527/22527 [35:24<00:00, 10.61it/s]


    Coreset done in 2124.0s
   RAM: 2.8/13.6 GB (26%) | VRAM: 0.38/15.64 GB

[7] Building memory bank...
  [metal_nut] Memory bank : (22528, 1536) | 132.0 MB
  [metal_nut] Saved to Drive : /content/drive/MyDrive/industrial_anomaly_project/models/patchcore_metal_nut.npy
  [metal_nut] Temp memmap deleted.
   RAM: 2.9/13.6 GB (26%) | VRAM: 0.38/15.64 GB

[8] Building FAISS index...
  FAISS index : IndexFlatL2 (exact)
  Vectors     : 22,528 | dim: 1536

[9] Anomaly scoring...
  [metal_nut] Test holdout images: 100


  Scoring [metal_nut]: 100%|██████████| 7/7 [01:15<00:00, 10.84s/it]


  [metal_nut] Scoring done in 75.8s (758.5 ms/img)

[10] Saving results...
  [metal_nut] Results saved:
     scores  : (100,) | range [13.1133, 39.3620]
     maps    : (100, 32, 32)
     labels  : (100,) | normal=17 defective=83

[Cleanup] metal_nut...
  [metal_nut] Memory cleared.

✅ METAL_NUT done in 37.1 min

════════════════════════════════════════════════════════════
  All categories processed.
════════════════════════════════════════════════════════════


---
### STEP 12 — Save Pipeline Metadata

In [17]:
# ─────────────────────────────────────────────────────────────────
# STEP 12 — Save pipeline metadata to Drive
# ─────────────────────────────────────────────────────────────────

META_PATH = os.path.join(RESULTS_DRIVE, 'patchcore_metadata.json')

with open(META_PATH, 'w') as f:
    json.dump(pipeline_metadata, f, indent=4)

print('✅ Pipeline metadata saved.\n')
print(f'   Path: {META_PATH}\n')

# ── Summary table ─────────────────────────────────────────────────
print(f'{"Category":<12} {"Train":<8} {"Patches":<12} {"Coreset":<10} {"Bank MB":<10} {"Test":<8} {"Time(min)"}')
print('─' * 75)
for cat, m in pipeline_metadata.items():
    print(
        f"{cat:<12} "
        f"{m['n_train_images']:<8} "
        f"{m['n_total_patches']:<12,} "
        f"{m['coreset_size']:<10,} "
        f"{m['memory_bank_mb']:<10} "
        f"{m['n_test_scored']:<8} "
        f"{m['elapsed_seconds']/60:.1f}"
    )
print('─' * 75)

✅ Pipeline metadata saved.

   Path: /content/drive/MyDrive/industrial_anomaly_project/results/patchcore_metadata.json

Category     Train    Patches      Coreset    Bank MB    Test     Time(min)
───────────────────────────────────────────────────────────────────────────
leather      245      250,880      25,088     147.0      104      46.6
tile         230      235,520      23,552     138.0      97       40.2
metal_nut    220      225,280      22,528     132.0      100      37.1
───────────────────────────────────────────────────────────────────────────


---
### STEP 13 — Final Verification

In [18]:
# ─────────────────────────────────────────────────────────────────
# STEP 13 — Final asset verification
# ─────────────────────────────────────────────────────────────────

print('🔍 Verifying all output files...\n')

expected_files = []
for cat in CATEGORIES:
    expected_files += [
        # Memory banks
        os.path.join(MODEL_DRIVE,   f'patchcore_{cat}.npy'),
        # Scores
        os.path.join(RESULTS_DRIVE, f'patchcore_{cat}_scores.npy'),
        os.path.join(RESULTS_DRIVE, f'patchcore_{cat}_maps.npy'),
        os.path.join(RESULTS_DRIVE, f'patchcore_{cat}_labels.npy'),
        os.path.join(RESULTS_DRIVE, f'patchcore_{cat}_meta.json'),
    ]
expected_files.append(META_PATH)

all_ok = True
for fp in expected_files:
    exists  = os.path.exists(fp)
    size_kb = os.path.getsize(fp) / 1024 if exists else 0
    icon    = '✅' if exists else '❌'
    print(f'  {icon} {os.path.basename(fp):<45} {size_kb:>8.1f} KB')
    if not exists:
        all_ok = False

if not all_ok:
    raise SystemExit('❌ Some output files are missing. Check errors above.')

# Final memory cleanup
clear_memory()

print()
print('═' * 60)
print('✅ Notebook 02 COMPLETE — PatchCore Pipeline Done')
print('═' * 60)
print()
print('📌 Assets saved to Drive:')
print(f'   Memory banks → {MODEL_DRIVE}')
print(f'   Scores/maps  → {RESULTS_DRIVE}')
print()
print('📌 Next step: Run 03_autoencoder_training.ipynb')

🔍 Verifying all output files...

  ✅ patchcore_leather.npy                         150528.1 KB
  ✅ patchcore_leather_scores.npy                       0.5 KB
  ✅ patchcore_leather_maps.npy                       416.1 KB
  ✅ patchcore_leather_labels.npy                       0.5 KB
  ✅ patchcore_leather_meta.json                        7.9 KB
  ✅ patchcore_tile.npy                            141312.1 KB
  ✅ patchcore_tile_scores.npy                          0.5 KB
  ✅ patchcore_tile_maps.npy                          388.1 KB
  ✅ patchcore_tile_labels.npy                          0.5 KB
  ✅ patchcore_tile_meta.json                           7.5 KB
  ✅ patchcore_metal_nut.npy                       135168.1 KB
  ✅ patchcore_metal_nut_scores.npy                     0.5 KB
  ✅ patchcore_metal_nut_maps.npy                     400.1 KB
  ✅ patchcore_metal_nut_labels.npy                     0.5 KB
  ✅ patchcore_metal_nut_meta.json                      8.0 KB
  ✅ patchcore_metadata.json          